# AutoSort: YOLO11 classification → ONNX
Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy từng ô. Tải `autosort-training.zip` ở máy lên khi được hỏi.
Dữ liệu calamansi công khai là dữ liệu tham khảo; kết quả test chưa đại diện cho camera thực tế. Ba ảnh mẫu chỉ dùng đối chiếu PyTorch–ONNX. Nguồn và giấy phép nằm trong `datasets/kumquat/ATTRIBUTION.txt`.


In [ ]:
from google.colab import files
from pathlib import Path
import json, os, subprocess, sys, zipfile
uploaded = files.upload()
bundle = next((Path(name) for name in uploaded if name.lower().endswith('.zip')), None)
if bundle is None:
    raise ValueError('Hãy chọn autosort-training.zip')
workdir = Path('/content/autosort-train').resolve()
workdir.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    for name in archive.namelist():
        if not (workdir / name).resolve().is_relative_to(workdir):
            raise ValueError('ZIP có đường dẫn không hợp lệ')
    archive.extractall(workdir)
os.chdir(workdir)
print('Working directory:', workdir)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'training/requirements.txt'], check=True)


In [ ]:
sys.path.insert(0, str(workdir / 'training'))
from dataset_checks import inspect_dataset
counts = inspect_dataset('datasets/kumquat')
print(json.dumps(counts, indent=2))
manifest_path = Path('datasets/kumquat/dataset_manifest.json')
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print('Source:', manifest['source_url'])
    print('License:', manifest['license'])
    print('Limitations:', manifest['limitations'])


In [ ]:
import torch
device = '0' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
subprocess.run([sys.executable, 'training/train.py', '--dataset', 'datasets/kumquat', '--device', device, '--epochs', '30', '--batch', '16', '--imgsz', '224'], check=True)
run = json.loads(Path('runs/autosort/latest_run.json').read_text())
print('Test Top-1:', run['test_top1'])
print('Best weights:', run['best_weights'])


In [ ]:
subprocess.run([sys.executable, 'training/export_onnx.py', '--weights', run['best_weights'], '--destination', 'cloud-ai/model', '--threshold', '0.70'], check=True)


In [ ]:
subprocess.run([sys.executable, 'training/check_parity.py', '--weights', run['best_weights'], '--onnx', 'cloud-ai/model/best.onnx', '--classes', 'cloud-ai/model/classes.json'], check=True)


In [ ]:
artifacts = Path('autosort-model-artifacts.zip')
with zipfile.ZipFile(artifacts, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(run['best_weights'], 'best.pt')
    for name in ('best.onnx', 'classes.json'):
        archive.write(Path('cloud-ai/model') / name, name)
    archive.write('runs/autosort/latest_run.json', 'training_result.json')
    for name in ('dataset_manifest.json', 'ATTRIBUTION.txt'):
        path = Path('datasets/kumquat') / name
        if path.exists():
            archive.write(path, name)
files.download(str(artifacts))


Sau khi tải xong, đưa `best.onnx` và `classes.json` vào `cloud-ai/model/`. Giữ `best.pt` để có thể kiểm tra lại. Chỉ triển khai sau khi parity đạt và đã đánh giá trên ảnh độc lập từ camera thực tế.
